# ATLAS Explorer: Multicore + Baseline Diff Report (Notebook)

Run a multicore experiment, then build a ZIP bundle with JSON, Markdown, and rich HTML reports. Optionally include a baseline diff.

## Prerequisites
- Configure credentials once: run the CLI configurator or set `MIPS_ATLAS_CONFIG`.
- Example CLI: `uv run atlasexplorer/atlasexplorer.py configure`.
- Ensure example ELFs exist (defaults use files in `resources/`).

In [ ]:
# Settings — adjust as needed
elfs = ["resources/mandelbrot_rv64_O0.elf", "resources/memcpy_rv64.elf"]
expdir = "myexperiments"
core = "I8500_(2_threads)"
channel = "development"
apikey = None   # or a string
region = None   # or a string
verbose = False

# Optional baseline summary.json (set to a path string to enable diffs)
baseline = None  # e.g., 'atlas-explorer-experiments/.../reports/summary/summary.json'
# Output ZIP bundle path
bundle_out = "diff_bundle.zip"

In [ ]:
# Environment setup
import os, locale, tempfile, zipfile
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
locale.setlocale(locale.LC_ALL, "")

config_env = os.environ.get("MIPS_ATLAS_CONFIG")
home_dir = os.path.expanduser("~")
config_file = os.path.join(home_dir, ".config", "mips", "atlaspy", "config.json")
if not apikey and not config_env and not os.path.exists(config_file):
    print("Tip: run 'uv run atlasexplorer/atlasexplorer.py configure' or set MIPS_ATLAS_CONFIG.")

In [ ]:
# Run the experiment
from atlasexplorer.atlasexplorer import AtlasExplorer, Experiment

aeinst = AtlasExplorer(apikey, channel, region, verbose=verbose)
experiment = Experiment(expdir, aeinst, verbose=verbose)
for elf in elfs:
    experiment.addWorkload(os.path.abspath(elf))
experiment.setCore(core)
experiment.run()

# Find newest summary.json
exp_path = Path(expdir)
summary_candidates = list(exp_path.rglob("summary/summary.json"))
if not summary_candidates:
    raise FileNotFoundError("No summary.json found under experiment directory")
summary_candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
summary_path = summary_candidates[0]
print(f"Produced summary: {summary_path}")

In [ ]:
# Build report bundle (JSON, MD, rich HTML), with optional baseline diff
from atlasexplorer.reporting.parser import parse_summary_json
from atlasexplorer.reporting.derive import apply_derivations
from atlasexplorer.reporting.thresholds import apply_thresholds
from atlasexplorer.reporting.export import export_json, export_markdown, export_rich_html

main_report = apply_thresholds(apply_derivations(parse_summary_json(str(summary_path))))
base_report = None
if baseline:
    base_report = apply_derivations(parse_summary_json(str(Path(baseline).resolve())))

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)
    json_path = tmpdir_path / 'report.json'
    md_path = tmpdir_path / 'report.md'
    rich_path = tmpdir_path / 'report_rich.html'
    export_json(main_report, json_path)
    export_markdown(main_report, md_path)
    export_rich_html(main_report, rich_path)

    if base_report:
        lines = ["# Baseline Comparison\n", "| Metric | New | Baseline | Δ | Δ% |", "|--------|----|----------|---|----|"]
        base_map = {m.name: m for m in base_report.metrics}
        for m in main_report.metrics:
            if m.name in base_map and m.value is not None and base_map[m.name].value is not None:
                new_v = m.value
                old_v = base_map[m.name].value
                delta = new_v - old_v
                pct = (delta / old_v * 100) if old_v not in (0, None) else float('inf')
                lines.append(f"| {m.name} | {new_v:.4g} | {old_v:.4g} | {delta:+.4g} | {pct:+.2f}% |")
        (tmpdir_path / 'baseline_diff.md').write_text("\n".join(lines))

    bundle_out_path = Path(bundle_out).resolve()
    bundle_out_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_out_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for f in tmpdir_path.iterdir():
            zf.write(f, arcname=f.name)

print(f"Bundle written to: {bundle_out_path}")

In [ ]:
# Quick ad-hoc analysis as a DataFrame
from atlasexplorer.reporting.parser import parse_summary_json
from atlasexplorer.reporting.derive import apply_derivations
import pandas as pd

report = apply_derivations(parse_summary_json(str(summary_path)))
df = pd.DataFrame([m.model_dump() for m in report.metrics])
df.head()